# 02 – Preprocessing and Isolation Forest prototype

Case study *Early Warning of Air-Leak Failures in Metro Compressors* (Group 6, MDM2, TH Deggendorf).

This notebook belongs to the methodology milestone. It runs the preprocessing pipeline and a first learned detector:

1. Remove frozen logging segments
2. Build gap-aware 10-minute windows
3. Chronological split: train = 1–21 February, validation = last week of February, test = March–August
4. Scale with training statistics only
5. Isolation Forest (Liu et al., 2008)
6. Common alarm logic: threshold Q3 + 3·IQR of the validation scores (Silva et al., 2023) and a low-pass filter (Ribeiro et al., 2016)
7. Event-level evaluation, also against the first LPS activation
8. Comparison with the threshold baseline, a random-alarm baseline and with/without frozen-segment masking

Code: `src/preprocessing.py`, `src/detectors.py`, `src/alarm_logic.py`, `src/evaluation.py`, `src/random_baseline.py`, `src/run_experiment.py`.

Data: Davari, N., Veloso, B., Ribeiro, R., & Gama, J. (2021). *MetroPT-3 Dataset*. UCI Machine Learning Repository. https://doi.org/10.24432/C5VW3R (CC BY 4.0). Run `python scripts/download_data.py` first, or use the Colab cell below.

In [ ]:
# Google Colab only: clone the repository and download the data
import os
import subprocess
import sys

BRANCH = "main"  # use a work-in-progress branch here to test unmerged code

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo_dir = "/content/IntelligentSystemsCaseStudy"
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", "--branch", BRANCH,
                        "https://github.com/mohamedfarez/IntelligentSystemsCaseStudy.git", repo_dir], check=True)
    os.chdir(repo_dir)
    subprocess.run([sys.executable, "scripts/download_data.py"], check=True)

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT.resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import src
from src import config
from src.data_loading import load_metropt3
from src.evaluation import evaluate_events, lps_onsets
from src.preprocessing import PreprocessConfig, prepare
from src.random_baseline import run_random_baseline
from src.run_experiment import run
from src.threshold_baseline import run_baseline

print("project code version:", src.__version__)
config.set_global_seed()
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160)

def save(fig, name):
    fig.savefig(config.figure_path(name), dpi=150, bbox_inches="tight")

def shade_failures(ax):
    for _, p in config.FAILURE_PERIODS.iterrows():
        ax.axvspan(p["start"], p["end"], color="red", alpha=0.25, lw=0)

In [ ]:
df = load_metropt3()
print(df.shape, df.index.min(), "->", df.index.max())

## 1. Frozen segments

Records in which all analogue signals stay exactly the same for at least 30 minutes are removed (see notebook 01, section 2b).

In [ ]:
pre_cfg = PreprocessConfig()
data = prepare(df, pre_cfg)
print(f"Frozen segments: {len(data.frozen_segments)}, records removed: {data.n_records_removed:,} "
      f"({100 * data.n_records_removed / len(df):.1f} %)")
data.frozen_segments

## 2. Gap-aware windows

Features per 10-minute window: mean, std, min and max of the 7 analogue signals, the share of active time of the digital signals (without LPS, which we keep for the evaluation), and the share of time under load. A window is kept only if it has at least 50 % of the expected records (60 at 10 s), so gaps are not filled with made-up values.

In [ ]:
feats = data.features
print(f"Windows kept: {len(feats):,} | features: {data.train.shape[1]}")
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(feats["coverage"], bins=50)
ax.set(xlabel="Share of expected records per window", ylabel="Windows", title="Window coverage")
save(fig, "prep_01_window_coverage")
plt.show()

## 3. Chronological split and 4. scaling

Normal period = windows ending on or before 1 March 2020. Its last 7 days are the validation set: the detector never trains on them, and the alarm threshold is computed from them (scores on the detector's own training data are too optimistic). The scaler is fitted on the training windows only.

If the test months behave like February, the scaled test features should have mean ≈ 0 and std ≈ 1. The plot shows how far each feature moves away from that (the February shift seen in notebook 01).

In [ ]:
for name, part in [("Train", data.train), ("Validation", data.val), ("Test", data.test)]:
    print(f"{name:<11}{len(part):>7,} windows  ({part.index.min()} -> {part.index.max()})")

shift = data.test.mean().abs().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
shift.head(15)[::-1].plot.barh(ax=ax)
ax.set(xlabel="|mean of scaled test feature|  (0 = same as February)", title="Largest shifts between train and test")
save(fig, "prep_02_feature_shift")
plt.show()

## 5. Isolation Forest and 6. alarm logic

In [ ]:
out = run(df, "iforest", pre_cfg)
rule = out["rule"]
print("Threshold (Q3 + 3*IQR of validation scores):", round(rule.threshold_, 4))
print("Test windows above threshold:", f"{(out['test_scores'] > rule.threshold_).mean():.1%}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(out["train_scores"], bins=80, alpha=0.6, density=True, label="train (1-21 Feb)")
ax.hist(out["val_scores"], bins=40, alpha=0.6, density=True, label="validation (22-28 Feb)")
ax.hist(out["test_scores"], bins=80, alpha=0.6, density=True, label="test (Mar-Sep)")
ax.axvline(rule.threshold_, color="k", ls="--", label="threshold")
ax.set(xlabel="Isolation Forest anomaly score", ylabel="Density")
ax.legend()
save(fig, "iforest_01_score_distribution")
plt.show()

In [ ]:
scores, alarms = out["test_scores"], out["alarms"]
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(scores.index, scores, lw=0.4, color="tab:blue", label="score")
ax.axhline(rule.threshold_, color="k", ls="--", lw=1, label="threshold")
ax.scatter(alarms.index[alarms], scores[alarms], s=4, color="tab:orange", label="alarm", zorder=3)
shade_failures(ax)
for _, s in data.frozen_segments.iterrows():
    ax.axvspan(s["start"], s["end"], color="grey", alpha=0.3, lw=0)
ax.set(title="Isolation Forest on the test period (red = reported failures, grey = removed frozen segments)", ylabel="score")
ax.legend(loc="upper left", markerscale=3)
save(fig, "iforest_02_alarms_test_period")
plt.show()

## 7. Event-level evaluation

`lead_time_min` is measured to the reported failure start; `lead_vs_lps_min` to the first LPS activation within [start − 2 h, end]. Positive values mean the alarm came earlier.

In [ ]:
out["events"]

In [ ]:
lps = lps_onsets(df)
print("First LPS activation per failure:")
print(lps.to_string())

## 8. Comparison

Same test period, same horizon (2 h), same event-level metrics. False alarms per day are counted per day of *observed* data, so recording gaps do not dilute the rate. The threshold baseline keeps its own rules (notebook 01); Isolation Forest uses the common alarm logic. `iforest_no_frozen_mask` repeats the Isolation Forest run without removing frozen segments.

**Random-alarm baseline:** 100 runs in which every test window exceeds the threshold at random with the same probability as the Isolation Forest, followed by the same low-pass filter. It shows what detection and lead time can be reached by chance with the same amount of alarms (Kim et al., 2022). The table shows the mean over the runs.

In [ ]:
base = run_baseline(df)
_, base_summary = evaluate_events(base["alarms"], "2h", "10min", lps=lps)
nomask = run(df, "iforest", PreprocessConfig(mask_frozen=False))
exceed_rate = float((out["test_scores"] > rule.threshold_).mean())
rand = run_random_baseline(out["test_scores"].index, exceed_rate, n_runs=100)

comparison = pd.DataFrame({
    "threshold_baseline": base_summary,
    "iforest": out["summary"],
    "iforest_no_frozen_mask": nomask["summary"],
    "random_mean_100_runs": rand.mean(),
}).round(3)
comparison

In [ ]:
# How often does chance alone look as good as the Isolation Forest?
print(f"Random runs detecting all 4 failures:      {(rand['n_detected'] == 4).mean():.0%}")
print(f"Random runs with at least 1 early warning:  {(rand['n_early_warnings'] >= 1).mean():.0%}")
print(f"Random runs with median lead >= Isolation Forest ({out['summary']['median_lead_time_min']:.0f} min): "
      f"{(rand['median_lead_time_min'] >= out['summary']['median_lead_time_min']).mean():.0%}")

In [ ]:
lead = pd.DataFrame({
    "baseline": base["events"].set_index("failure_id")["lead_time_min"],
    "iforest": out["events"].set_index("failure_id")["lead_time_min"],
    "iforest_no_mask": nomask["events"].set_index("failure_id")["lead_time_min"],
})
lead

In [ ]:
out["events"].to_csv(config.RESULTS_DIR / "iforest_events.csv", index=False)
pd.Series(out["summary"]).to_csv(config.RESULTS_DIR / "iforest_summary.csv", header=["value"])
comparison.to_csv(config.RESULTS_DIR / "detector_comparison.csv")
lead.to_csv(config.RESULTS_DIR / "lead_time_comparison.csv")
rand.to_csv(config.RESULTS_DIR / "random_baseline_runs.csv", index=False)
print("saved to", config.RESULTS_DIR)

## 9. Notes

No parameter was tuned on the test period. Next steps: handling of the February shift and feature selection (decided on February data only), the LSTM autoencoder, SHAP explanations and sensitivity analyses.